# 465. 最优账单平衡：贪心踩坑、DFS 与状态压缩 DP

[原文](https://leetcode.cn/problems/optimal-account-balancing/solution/465qian-du-huan-qing-liao-li-kou-huan-bu-eszd/)

## 错误贪心（仅供复盘）

排序抵消能结清，但不保证最少次数；原文反例返回 6，最优答案为 4。

```python
#1. Greedy？

from typing import List
from collections import defaultdict



class Solution:
    def minTransfers(self, transactions: List[List[int]]) -> int:
        record = defaultdict(int)
        for source, target, amount in transactions:
            record[source] += amount
            record[target] -= amount
        print(record)

        pos = sorted([v for v in record.values() if v > 0], reverse=True)
        neg = sorted([v for v in record.values() if v < 0], reverse=False)

        ans = 0
        res = 0
        pos_idx = 0
        neg_idx = 0

        print(pos)
        print(neg)

        while pos_idx < len(pos) or neg_idx < len(neg):
            # 余额为 0 时，取一个正余额开始新一组
            if res == 0:
                res = pos[pos_idx]
                pos_idx += 1

            # 当前负余额还没抵消完，加入一个正余额
            while res < 0:
                res += pos[pos_idx]
                pos_idx += 1
                ans += 1

            # 当前正余额还没抵消完，加入一个负余额
            while res > 0:
                res += neg[neg_idx]
                neg_idx += 1
                ans += 1

        return ans
```

### 早期错误片段（不执行）

余额为零时没有推进，会死循环；负余额应相加，不能再相减。

```python
res = 0
while pos_idx < len(pos):
    while res < 0:
        ...
    while res > 0:
        ...
```

```python
res -= neg[neg_idx]  # neg 本来就是负数，减掉它反而会变大
```

## DFS：枚举选人顺序

In [ ]:
# res 是已选余额之和；加入下一人前 res 非零才增加一次交易。
# 分支独立计算 new_cnt，避免同层累计计数。

In [ ]:
# 2. DFS
from typing import List
from collections import defaultdict

class Solution:
    def minTransfers(self, transactions: List[List[int]]) -> int:
        record = defaultdict(int)
        ans = float('inf')
        for source, target, amount in transactions:
            record[source] += amount
            record[target] -= amount
        print(record)
        debt = [amount for amount in record.values() if amount != 0]
        print(debt)

        def dfs(res,debt, cnt):
            nonlocal ans
            # print(debt)
            if not debt and res == 0:
                ans = min(cnt, ans)
                return
            else:
                for i in range(len(debt)):
                    new_cnt = cnt
                    if res!=0:
                        new_cnt = cnt+1
                    dfs(res+debt[i], debt[:i]+debt[i+1:] if len(debt) else [], new_cnt)
        dfs(0,debt,0)
        return ans

### 早期错误计数片段（不执行）

在循环中修改 cnt，会让后续分支继承前一分支的增量。

```python
for i in range(len(debt)):
    if res != 0:
        cnt += 1
    dfs(..., cnt)
```

## DFS Pruning：删去 res，按当前次数剪枝

In [ ]:
# 总余额为零，因此 res = -sum(debt)，无需重复传参。
# 原版 cnt > ans 保留原样；等号情形仍会继续搜索。

In [ ]:
# 3. DFS Pruning
from typing import List
from collections import defaultdict

class Solution:
    def minTransfers(self, transactions: List[List[int]]) -> int:
        record = defaultdict(int)
        ans = float('inf')
        for source, target, amount in transactions:
            record[source] += amount
            record[target] -= amount
        print(record)
        debt = [amount for amount in record.values() if amount != 0]
        print(debt)

        def dfs(debt, cnt):
            nonlocal ans
            # print(debt)
            if not debt:
                ans = min(cnt, ans)
                return

            elif cnt> ans:
                return

            else:
                for i in range(len(debt)):
                    new_cnt = cnt
                    if sum(debt)!=0:
                        new_cnt = cnt+1
                    dfs(debt[:i]+debt[i+1:] if len(debt) else [], new_cnt)
        dfs(debt,0)
        return ans

## DFS Pruning 2：同层去重

In [ ]:
# 相同余额在同一层只选一次；排序后删除元素，仍保持有序。

In [ ]:
# 3. DFS Pruning 2
from typing import List
from collections import defaultdict

class Solution:
    def minTransfers(self, transactions: List[List[int]]) -> int:
        record = defaultdict(int)
        ans = float('inf')
        for source, target, amount in transactions:
            record[source] += amount
            record[target] -= amount
        print(record)
        debt = sorted([amount for amount in record.values() if amount != 0])
        print(debt)

        def dfs(debt, cnt):
            nonlocal ans
            # print(debt)
            if not debt:
                ans = min(cnt, ans)
                return

            elif cnt> ans:
                return

            else:
                new_cnt = cnt
                if sum(debt)!=0:
                    new_cnt = cnt+1
                visited = set()
                for i in range(len(debt)):
                    if debt[i] in visited:
                        continue
                    next_debt = debt[:i]+debt[i+1:]

                    visited.add(debt[i])
                    dfs(next_debt if len(debt) else [], new_cnt)
        dfs(debt,0)
        return ans

## DFS Pruning 3：缓存 (debt, cnt)

In [ ]:
# 元组可作为缓存键；历史次数 cnt 仍在键里，合并重复状态不充分。

In [ ]:
# 3. DFS Pruning 3
from typing import List
from collections import defaultdict
from functools import lru_cache

class Solution:
    def minTransfers(self, transactions: List[List[int]]) -> int:
        record = defaultdict(int)
        ans = float('inf')
        for source, target, amount in transactions:
            record[source] += amount
            record[target] -= amount
        debt = tuple(sorted([amount for amount in record.values() if amount != 0]))

        @lru_cache(None)
        def dfs(debt, cnt):
            nonlocal ans
            # print(debt)
            if not debt:
                ans = min(cnt, ans)
                return

            elif cnt> ans:
                return

            else:
                new_cnt = cnt
                if sum(debt)!=0:
                    new_cnt = cnt+1
                visited = set()
                for i in range(len(debt)):
                    if debt[i] in visited:
                        continue
                    next_debt = debt[:i]+debt[i+1:]

                    visited.add(debt[i])
                    dfs(next_debt if len(debt) else [], new_cnt)
        dfs(debt,0)
        return ans

## DFS Pruning 4：缓存剩余任务的答案

In [ ]:
# dfs(debt) 返回当前剩余状态最少还需要增加的交易次数。
# 当前剩余余额和非零时 cost = 1；空状态返回 0。

In [ ]:
# 3. DFS Pruning 4
from typing import List
from collections import defaultdict
from functools import lru_cache

class Solution:
    def minTransfers(self, transactions: List[List[int]]) -> int:
        record = defaultdict(int)
        for source, target, amount in transactions:
            record[source] += amount
            record[target] -= amount
        debt = tuple(sorted([amount for amount in record.values() if amount != 0]))

        @lru_cache(None)
        def dfs(debt):
            if not debt:
                return 0
            else:
                cost = int(sum(debt)!=0)
                visited = set()
                best = float('inf')
                for i in range(len(debt)):
                    if debt[i] in visited:
                        continue
                    visited.add(debt[i])
                    next_debt = debt[:i] + debt[i+1:]
                    best = min(dfs(next_debt)+cost,best)
            return best
                
        return dfs(debt)

## 状态压缩 DP（最终版）

In [ ]:
# mask 的置位表示还未选择的人；清掉一位后的状态更小，按 mask 递增填表。
# sums[mask] 非零时本次计数加一，与记忆化搜索的转移一致。

In [ ]:
from typing import List
from collections import defaultdict

class Solution:
    def minTransfers(self, transactions: List[List[int]]) -> int:
        record = defaultdict(int)
        for source, target, amount in transactions:
            record[source] += amount
            record[target] -= amount

        debt = [amount for amount in record.values() if amount != 0]
        n = len(debt)
        size = 1 << n

        # sums[mask]：剩余这些人的余额之和
        sums = [0] * size

        # dp[mask]：剩余这些人时，最少还需要增加的交易次数
        dp = [float('inf')] * size
        dp[0] = 0

        for mask in range(1, size):
            bit = mask & -mask
            i = bit.bit_length() - 1
            sums[mask] = sums[mask ^ bit] + debt[i]

            cost = int(sums[mask] != 0)

            for i in range(n):
                if mask & (1 << i):
                    next_mask = mask ^ (1 << i)
                    dp[mask] = min(dp[mask], dp[next_mask] + cost)

        return dp[size - 1]


## 原文样例（运行最终版）

In [ ]:
transactions = [[1,5,8],[8,9,8],[2,3,9],[4,3,1]]
solution = Solution()
ans = solution.minTransfers(transactions)
print(ans)